# Lab 2 — Oxford Pet Semantic Segmentation
U-Net / ResNet34-UNet 訓練與推論

## 使用前準備
1. 執行階段 → 變更執行階段類型 → **GPU**
2. Google Drive 的 `MyDrive/lab2/` 裡要有：
   ```
   MyDrive/lab2/
   ├── dataset/oxford-iiit-pet/       ← 圖片和 trimaps（只需上傳一次）
   ├── nycu-2026-spring-dl-lab2-unet/ ← split txt 檔
   └── binary-semantic-segmentation-res-net-34-u-net/ ← split txt 檔
   ```
3. 程式碼從 GitHub clone，**不需要上傳 src/**

In [ ]:
# ── 1. 確認 GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Clone repo（只拿程式碼，很快）────────────────────────────────
import os

if not os.path.exists('/content/NYCU-Deep-Learing'):
    !git clone https://github.com/InnisChen/NYCU-Deep-Learing /content/NYCU-Deep-Learing
else:
    print('Repo already cloned, pulling latest...')
    !git -C /content/NYCU-Deep-Learing pull

%cd /content/NYCU-Deep-Learing/lab2
import sys
sys.path.insert(0, 'src')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 3. 掛載 Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 4. Symlink dataset 和 split 資料夾（不複製，直接指向 Drive）──────
DRIVE_LAB2 = '/content/drive/MyDrive/lab2'

links = {
    f'{DRIVE_LAB2}/dataset':                                          'dataset',
    f'{DRIVE_LAB2}/nycu-2026-spring-dl-lab2-unet':                   'nycu-2026-spring-dl-lab2-unet',
    f'{DRIVE_LAB2}/binary-semantic-segmentation-res-net-34-u-net':   'binary-semantic-segmentation-res-net-34-u-net',
}

for src, dst in links.items():
    if os.path.exists(dst) or os.path.islink(dst):
        print(f'  already exists: {dst}')
    elif os.path.exists(src):
        os.symlink(src, dst)
        print(f'  linked: {dst} → {src}')
    else:
        print(f'  WARNING: source not found: {src}')

In [ ]:
# ── 5. 確認資料集結構 ─────────────────────────────────────────────────
dataset_root = 'dataset/oxford-iiit-pet'
print('Images :', len(os.listdir(f'{dataset_root}/images')), 'files')
print('Trimaps:', len(os.listdir(f'{dataset_root}/annotations/trimaps')), 'files')

for split_dir in ['nycu-2026-spring-dl-lab2-unet', 'binary-semantic-segmentation-res-net-34-u-net']:
    if os.path.exists(split_dir):
        files_found = os.listdir(split_dir)
        print(f'{split_dir}/: {files_found}')
    else:
        print(f'MISSING: {split_dir}/')

## 6. 更新程式碼
本機改完程式碼 → `git push` → 執行下面這個 cell

In [ ]:
# ── 6. 拉最新程式碼 ───────────────────────────────────────────────────
!git pull
!git log --oneline -3

## 7. 訓練

In [ ]:
# ── 7. 設定訓練參數 ───────────────────────────────────────────────────
MODEL      = 'unet'          # 'unet' 或 'resnet34_unet'
EPOCHS     = 50
BATCH_SIZE = 16              # T4 GPU 建議 16~32
LR         = 1e-3
DATA_PATH  = 'dataset/oxford-iiit-pet'
SAVE_PATH  = '/content/drive/MyDrive/lab2/saved_models'  # 直接存到 Drive

# split_dir 依 model 自動決定（train.py 尚未自動選，這裡明確指定）
SPLIT_MAP = {
    'unet':          'nycu-2026-spring-dl-lab2-unet',
    'resnet34_unet': 'binary-semantic-segmentation-res-net-34-u-net',
}
SPLIT_DIR = SPLIT_MAP[MODEL]

cmd = (f'python src/train.py'
       f' --model {MODEL}'
       f' --split_dir {SPLIT_DIR}'
       f' --data_path {DATA_PATH}'
       f' --save_path {SAVE_PATH}'
       f' --epochs {EPOCHS}'
       f' --batch_size {BATCH_SIZE}'
       f' --learning_rate {LR}')
print('Command:', cmd)

In [ ]:
# ── 執行訓練（結果直接存進 Drive，不會因 session 結束消失）──────────
import os
os.makedirs(SAVE_PATH, exist_ok=True)
!{cmd}

## 8. 推論（生成 submission.csv）

In [ ]:
# ── 8. 推論設定 ───────────────────────────────────────────────────────
MODEL      = 'unet'          # 與訓練時一致
DATA_PATH  = 'dataset/oxford-iiit-pet'
WEIGHT     = f'/content/drive/MyDrive/lab2/saved_models/{MODEL}_best.pth'
OUTPUT     = f'submission_{MODEL}.csv'
BATCH_SIZE = 16

cmd_inf = (f'python src/inference.py'
           f' --model {MODEL}'
           f' --data_path {DATA_PATH}'
           f' --weight {WEIGHT}'
           f' --output {OUTPUT}'
           f' --batch_size {BATCH_SIZE}')
print('Command:', cmd_inf)

In [ ]:
# ── 執行推論 ──────────────────────────────────────────────────────────
!{cmd_inf}

In [ ]:
# ── 確認並下載 submission.csv ─────────────────────────────────────────
import pandas as pd
from google.colab import files

df = pd.read_csv(OUTPUT)
print(f'Submission shape: {df.shape}')
print(df.head())

# 存到 Drive
import shutil
shutil.copy(OUTPUT, f'/content/drive/MyDrive/lab2/{OUTPUT}')
print(f'Saved to Drive')

# 下載到本機
files.download(OUTPUT)